In [38]:
import os
import shutil
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, Flatten, Dropout
from keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.optimizers import RMSprop
from keras.callbacks import EarlyStopping
from keras.regularizers import l2

In [39]:
!wget --no-check-certificate "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip" -O "/tmp/cats-and-dogs.zip"

--2026-07-28 14:40:13--  https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
Resolving download.microsoft.com (download.microsoft.com)... 23.61.214.15, 2600:1406:5e00:389::317f, 2600:1406:5e00:392::317f
Connecting to download.microsoft.com (download.microsoft.com)|23.61.214.15|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824887076 (787M) [application/octet-stream]
Saving to: ‘/tmp/cats-and-dogs.zip’

/tmp/cats-and-dogs. 100%[===================>] 786.67M   134MB/s    in 8.2s    

2026-07-28 14:40:21 (96.2 MB/s) - ‘/tmp/cats-and-dogs.zip’ saved [824887076/824887076]



In [40]:
def get_valid(file_path):
    correct_files = []
    for name in os.listdir(file_path):
        try:
            img = Image.open(file_path + "/" + name)
            correct_files.append(name)
        except UnidentifiedImageError:
            pass
    return correct_files

In [41]:
# Ścieżka do archiwum ZIP zawierającego obrazy kotów i psów.
local_zip = "/tmp/cats-and-dogs.zip"

# Otwiera archiwum ZIP w trybie odczytu.
zip_ref = zipfile.ZipFile(local_zip, "r")

# Wypakowuje całą zawartość archiwum do katalogu /tmp.
zip_ref.extractall("/tmp")

# Zamyka archiwum i zwalnia związane z nim zasoby.
zip_ref.close()


# Lista nazw klas, które będzie rozpoznawał model.
classes = ["Cat", "Dog"]

# Ścieżka do katalogu zawierającego oryginalne obrazy kotów.
original_cat_path = "/tmp/PetImages/Cat"

# Ścieżka do katalogu zawierającego oryginalne obrazy psów.
original_dog_path = "/tmp/PetImages/Dog"

# Pobiera nazwy prawidłowych plików z obrazami kotów.
# Funkcja get_valid() musi zostać wcześniej zdefiniowana.
original_cat = get_valid(original_cat_path)

# Pobiera nazwy prawidłowych plików z obrazami psów.
original_dog = get_valid(original_dog_path)


# Ustawia ziarno losowości, dzięki czemu wyniki mieszania będą powtarzalne.
random.seed(101)

# Losowo zmienia kolejność plików z obrazami kotów.
random.shuffle(original_cat)

# Losowo zmienia kolejność plików z obrazami psów.
random.shuffle(original_dog)


# Wybiera liczebność mniejszej klasy.
# Dzięki temu liczba wykorzystanych kotów i psów będzie jednakowa.
size = min(len(original_cat), len(original_dog))

# Przeznacza 70% obrazów każdej klasy na zbiór treningowy.
train_size = int(np.floor(0.7 * size))

# Przeznacza 20% obrazów każdej klasy na zbiór walidacyjny.
valid_size = int(np.floor(0.2 * size))

# Pozostałe 10% obrazów przeznacza na zbiór testowy.
# Takie obliczenie zapewnia, że suma podzbiorów będzie równa size.
test_size = size - train_size - valid_size


# Nazwa głównego katalogu, w którym powstanie przygotowany zbiór danych.
base_directory = "dataset"

# Tworzy główny katalog dataset.
# Polecenie zgłosi FileExistsError, jeśli katalog już istnieje.
os.mkdir(base_directory)

# Lista rodzajów podzbiorów danych.
type_datasets = ["train", "valid", "test"]

# Tworzy pusty słownik, w którym zostaną zapisane ścieżki do katalogów.
directories = {}


# Wykonuje pętlę dla zbioru treningowego, walidacyjnego i testowego.
for type_dataset in type_datasets:

    # Tworzy ścieżkę, np. dataset/train.
    directory = os.path.join(base_directory, type_dataset)

    # Tworzy katalog odpowiedniego podzbioru.
    os.mkdir(directory)

    # Wykonuje pętlę dla klasy Cat i klasy Dog.
    for name_class in classes:

        # Tworzy ścieżkę, np. dataset/train/Cat.
        animal = os.path.join(directory, name_class)

        # Tworzy katalog przeznaczony na obrazy danej klasy.
        os.mkdir(animal)

        # Zapisuje ścieżkę w słowniku, np. pod kluczem train_Cat.
        directories[f"{type_dataset}_{name_class}"] = animal + "/"


# Ustawia indeks pierwszej pary obrazów na 0.
index = 0

# Jednocześnie przechodzi po obrazach kotów i psów.
# zip() kończy działanie po wyczerpaniu krótszej listy.
for name_cat, name_dog in zip(original_cat, original_dog):

    # Pierwsze 70% obrazów przypisuje do zbioru treningowego.
    if index < train_size:
        type_of_dataset = "train"

    # Kolejne 20% obrazów przypisuje do zbioru walidacyjnego.
    elif index < train_size + valid_size:
        type_of_dataset = "valid"

    # Pozostałe obrazy przypisuje do zbioru testowego.
    else:
        type_of_dataset = "test"

    # Kopiuje bieżący obraz kota do odpowiedniego katalogu docelowego.
    shutil.copyfile(
        src=os.path.join(original_cat_path, name_cat),
        dst=os.path.join(directories[f"{type_of_dataset}_Cat"], name_cat)
    )

    # Kopiuje bieżący obraz psa do odpowiedniego katalogu docelowego.
    shutil.copyfile(
        src=os.path.join(original_dog_path, name_dog),
        dst=os.path.join(directories[f"{type_of_dataset}_Dog"], name_dog)
    )

    # Zwiększa indeks przed przejściem do następnej pary obrazów.
    index += 1


# Wyświetla liczbę psów i kotów w zbiorze treningowym.
print(
    f'Dog - train: {len(os.listdir(directories["train_Dog"]))}\t'
    f'Cat - train: {len(os.listdir(directories["train_Cat"]))}'
)

# Wyświetla liczbę psów i kotów w zbiorze walidacyjnym.
print(
    f'Dog - valid: {len(os.listdir(directories["valid_Dog"]))}\t'
    f'Cat - valid: {len(os.listdir(directories["valid_Cat"]))}'
)

# Wyświetla liczbę psów i kotów w zbiorze testowym.
print(
    f'Dog - test:  {len(os.listdir(directories["test_Dog"]))}\t'
    f'Cat - test:  {len(os.listdir(directories["test_Cat"]))}'
)

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


FileExistsError: [Errno 17] File exists: 'dataset'

Czas na eksploracyjną analizę danych – box ploty, wykresy punktowe, korelacje… to nie tutaj ;) Przede wszystkim zerknijmy na zdjęcia znajdujące się w zbiorze treningowym. W pierwszych dwóch wierszach znajdują się koty, a w kolejnych psy – zresztą dobrze to widać.

In [ ]:
# Tworzy figurę o rozmiarze 12 × 12 cali.
fig = plt.figure(figsize=(12, 12))

# Ustawia odstępy między wykresami:
# hspace odpowiada za odstęp pionowy, a wspace za odstęp poziomy.
fig.subplots_adjust(hspace=0.1, wspace=0.1)


# Pobiera osiem pierwszych plików z katalogu treningowego kotów.
# enumerate() zwraca jednocześnie indeks i nazwę pliku.
for i, element in enumerate(
    os.listdir(directories["train_Cat"])[:8]
):
    # Dodaje kolejny wykres do siatki składającej się z 4 wierszy i 4 kolumn.
    # Koty zajmują pozycje od 1 do 8.
    ax = fig.add_subplot(4, 4, i + 1)

    # Buduje pełną ścieżkę i otwiera obraz kota.
    img = Image.open(
        os.path.join(directories["train_Cat"], element)
    )

    # Wyświetla obraz na aktualnym wykresie.
    ax.imshow(img)

    # Ukrywa górną krawędź wykresu.
    ax.spines["top"].set_visible(False)

    # Ukrywa lewą krawędź wykresu.
    ax.spines["left"].set_visible(False)

    # Ukrywa dolną krawędź wykresu.
    ax.spines["bottom"].set_visible(False)

    # Ukrywa prawą krawędź wykresu.
    ax.spines["right"].set_visible(False)

    # Usuwa oznaczenia

Taka eksploracja na ten moment nam wystarczy. Zacznijmy trenować modele, każdy kolejny będzie w teorii bardziej zaawansowany niż poprzedni.

In [ ]:
# Ustawia szerokość i wysokość obrazów przekazywanych do modelu.
# Wszystkie obrazy zostaną przeskalowane do rozmiaru 150 × 150 pikseli.
img_width, img_height = 150, 150

# Ścieżka do katalogu zawierającego treningowe obrazy kotów i psów.
train_data_dir = "dataset/train/"

# Ścieżka do katalogu zawierającego obrazy walidacyjne.
validation_data_dir = "dataset/valid/"

# Maksymalna liczba epok treningu.
# Wartość 1000 jest bardzo wysoka, dlatego warto zastosować EarlyStopping.
epochs = 1000

# Liczba obrazów przetwarzanych przed każdą aktualizacją wag modelu.
batch_size = 64

# Liczba partii treningowych przetwarzanych w jednej epoce.
steps_per_epoch = train_size // batch_size

# Liczba partii walidacyjnych sprawdzanych po każdej epoce.
validation_steps = valid_size // batch_size

# Liczba epok bez poprawy wyniku, po których EarlyStopping przerwie trening.
patience = 5


# Tworzy generator podstawowych danych treningowych.
# rescale dzieli wartości pikseli przez 255, zmieniając zakres z 0–255 na 0–1.
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

# Tworzy generator danych walidacyjnych.
# Dane walidacyjne są tylko normalizowane — nie stosujemy na nich augmentacji.
validation_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)


# Odczytuje obrazy treningowe z katalogów dataset/train/Cat i dataset/train/Dog.
train_generator = train_datagen.flow_from_directory(
    # Główny katalog zbioru treningowego.
    train_data_dir,

    # Każdy obraz zostanie zmieniony do rozmiaru 150 × 150 pikseli.
    target_size=(img_height, img_width),

    # Generator będzie zwracał po 64 obrazy.
    batch_size=batch_size,

    # Klasy zostaną zapisane jako wartości binarne: 0 lub 1.
    class_mode="binary",

    # Losowo zmienia kolejność obrazów treningowych.
    shuffle=True,

    # Zapewnia powtarzalność losowego mieszania.
    seed=101
)


# Odczytuje obrazy walidacyjne z odpowiednich podkatalogów.
validation_generator = validation_datagen.flow_from_directory(
    # Główny katalog zbioru walidacyjnego.
    validation_data_dir,

    # Zmienia rozmiar każdego obrazu do 150 × 150 pikseli.
    target_size=(img_height, img_width),

    # Zwraca po 64 obrazy w każdej partii.
    batch_size=batch_size,

    # Zwraca klasy jako wartości 0 lub 1.
    class_mode="binary",

    # Nie miesza danych, co ułatwia późniejsze porównanie predykcji z etykietami.
    shuffle=False
)


# Tworzy generator treningowy stosujący augmentację.
# Augmentacja tworzy zmodyfikowane warianty obrazów podczas treningu.
train_datagen_augmentation = ImageDataGenerator(
    # Normalizuje wartości pikseli do zakresu 0–1.
    rescale=1.0 / 255,

    # Losowo pochyla obrazy maksymalnie o współczynnik 0.2.
    shear_range=0.2,

    # Losowo przybliża lub oddala obrazy.
    zoom_range=0.2,

    # Losowo obraca obrazy maksymalnie o 30 stopni.
    rotation_range=30,

    # Losowo odbija obrazy w poziomie.
    horizontal_flip=True
)


# Odczytuje dane treningowe i stosuje do nich losowe przekształcenia.
train_generator_augmentation = train_datagen_augmentation.flow_from_directory(
    # Główny katalog danych treningowych.
    train_data_dir,

    # Docelowy rozmiar obrazów.
    target_size=(img_height, img_width),

    # Liczba obrazów w jednej partii.
    batch_size=batch_size,

    # Problem binarny: Cat albo Dog.
    class_mode="binary",

    # Miesza kolejność obrazów podczas treningu.
    shuffle=True,

    # Ustawia ziarno losowości.
    seed=101
)

Powyżej ustaliśmy zmienne, które będą wykorzystane dla każdego modelu. Wskazaliśmy źródła dla naszych plików, rozmiar zdjęcia wejściowego, cierpliwość dla Early stopping czy przygotowanie danych. Każde zdjęcie jest standaryzowane przez dzielenie przez 255, ponieważ piksele mają wartość od 0 do 255. Ponadto dla zbioru treningowego również stosowana jest augmentacja.

Stwórzmy jeszcze listę, na którą będziemy dodawać nazwy modeli oraz katalogi, do których będziemy zapisywali wykresy oraz arkusze kalkulacyjne z ewaluacją.

In [ ]:
models = []
os.mkdir("history")
os.mkdir("charts")

Model podstawowy - **Baseline**

Pierwszy model to jest sieć poznana w poprzednim module, nie ma tu zatem warstw splotowych. Rozmiar wejściowy ustalono już wcześniej na 150(szerokość)x150(wysokość)x3(liczba kanałów – rgb). Cierpliwość przyjęto na 5 epok. Jeśli błąd na zbiorze walidacyjnym po tym czasie nie zmaleje, to model przestaje się uczyć. Jest to dobra praktyka, aby z jednej strony nie uczyć modelu bez potrzeby, a z drugiej strony nie dopuścić do sytuacji, w której model uczy się zbyt krótko. Jako funkcję aktywacji zastosowano ReLU, a w warstwie wyjściowej funkcję sigmoidalną. Metryką oceny jest dokładność. W celu skrócenia czasu uczenia zastosowano optymalizator w postaci algorytmu RMSProp. Zachęcamy do przeczytania o nim w dokumentacji. Szybkość uczenia została wybrana jako wartość 1e-4. Celem sieci jest klasyfikowanie, więc jako funkcję straty wybrano binarną entropię krzyżową.

In [ ]:
# Utworzenie bazowego modelu sekwencyjnego
model_baseline = Sequential()

# Spłaszczenie obrazu 150×150 RGB do jednowymiarowego wektora
model_baseline.add(Flatten(input_shape=(150, 150, 3)))

# Warstwa wyjściowa zwracająca prawdopodobieństwo jednej z dwóch klas
model_baseline.add(Dense(units=1, activation='sigmoid'))

# Konfiguracja modelu do klasyfikacji binarnej
model_baseline.compile(
    loss='binary_crossentropy',             # Funkcja straty dla dwóch klas
    optimizer=RMSprop(learning_rate=1e-4),  # Optymalizator i tempo uczenia
    metrics=['accuracy']                    # Metryka skuteczności modelu
)

# Wyświetlenie architektury i liczby parametrów modelu
model_baseline.summary()

# Dodanie nazwy modelu do listy porównywanych modeli
models.append("baseline")

Interpretacja wyników model_baseline.summary():
Warstwa Flatten przekształca obraz o wymiarach 150 × 150 × 3 w wektor zawierający 67 500 wartości:
150 × 150 × 3 = 67 500
Nie ma parametrów, ponieważ jedynie zmienia kształt danych.

Warstwa Dense zawiera jeden neuron wyjściowy. Ma 67 501 parametrów:
67 500 wag + 1 parametr bias = 67 501

Wszystkie 67 501 parametrów podlega uczeniu (Trainable params).

Model nie ma parametrów zamrożonych (Non-trainable params: 0).

Rozmiar parametrów modelu wynosi około 263,68 KB.

Wartość None w Output Shape oznacza, że model może przyjmować partie obrazów o dowolnej liczbie elementów. Jest to bardzo prosty model bazowy do klasyfikacji binarnej — nie analizuje jeszcze lokalnych cech obrazu, tak jak robią to warstwy konwolucyjne.

Ten kod tworzy mechanizm wczesnego zatrzymania uczenia:

monitor='val_accuracy' – obserwuje dokładność na zbiorze walidacyjnym.
patience=patience – przerwie uczenie, jeśli val_accuracy nie poprawi się przez określoną liczbę kolejnych epok. Wartość musi być wcześniej zapisana w zmiennej patience, np. patience = 5.
restore_best_weights=True – po zatrzymaniu przywróci wagi z epoki, w której model uzyskał najlepszą dokładność walidacyjną.

In [ ]:
es = EarlyStopping(
    patience=patience,
    monitor='val_accuracy',
    restore_best_weights=True
)

In [ ]:
import os  # Obsługa katalogów i ścieżek
import pandas as pd  # Praca z danymi tabelarycznymi

history_baseline = model_baseline.fit(  # Rozpoczęcie trenowania modelu
    train_generator,  # Generator danych treningowych
    steps_per_epoch=steps_per_epoch,  # Liczba partii danych w jednej epoce
    epochs=epochs,  # Maksymalna liczba epok
    validation_data=validation_generator,  # Generator danych walidacyjnych
    validation_steps=validation_steps,  # Liczba partii używanych do walidacji
    callbacks=[es]  # Zastosowanie mechanizmu EarlyStopping
)

history_baseline_df = pd.DataFrame(history_baseline.history)  # Zamiana historii uczenia na tabelę
os.makedirs('history', exist_ok=True)  # Utworzenie katalogu, jeśli nie istnieje
history_baseline_csv_file = 'history/history_baseline.csv'  # Określenie ścieżki pliku wynikowego
history_baseline_df.to_csv(history_baseline_csv_file, index=False)  # Zapis historii do pliku CSV

history_baseline.history zawiera zwykle wartości z każdej epoki, na przykład:
loss – strata na zbiorze treningowym,
accuracy – dokładność treningowa,
val_loss – strata walidacyjna,
val_accuracy – dokładność walidacyjna.

In [ ]:
import os  # Obsługa katalogów
import numpy as np  # Operacje numeryczne
import pandas as pd  # Odczytywanie plików CSV
import matplotlib.pyplot as plt  # Tworzenie wykresów

os.makedirs('charts', exist_ok=True)  # Utworzenie katalogu na wykresy, jeśli nie istnieje

max_index = 0  # Największa liczba epok spośród wszystkich modeli
min_accuracy = 1  # Najmniejsza dokładność spośród wszystkich modeli
max_loss = 0  # Największa wartość straty spośród wszystkich modeli
colors = plt.cm.rainbow(np.linspace(0, 1, len(models)))  # Wygenerowanie osobnego koloru dla każdego modelu

# Ustalenie wspólnych zakresów osi dla wszystkich wykresów
for model in models:  # Przejście przez nazwy wszystkich modeli
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii uczenia danego modelu
    df.index += 1  # Numerowanie epok od 1 zamiast od 0

    if max_index < df.index.max():  # Sprawdzenie, czy model trenował dłużej niż poprzednie
        max_index = df.index.max()  # Zapisanie największej liczby epok

    current_min_accuracy = df[['accuracy', 'val_accuracy']].min().min()  # Najmniejsza dokładność modelu
    if min_accuracy > current_min_accuracy:  # Sprawdzenie, czy znaleziono mniejszą dokładność
        min_accuracy = current_min_accuracy  # Aktualizacja dolnej granicy wykresów dokładności

    current_max_loss = df[['loss', 'val_loss']].max().max()  # Największa strata modelu
    if max_loss < current_max_loss:  # Sprawdzenie, czy znaleziono większą stratę
        max_loss = current_max_loss  # Aktualizacja górnej granicy wykresów straty


# Utworzenie osobnych wykresów dla każdego modelu
for model in models:  # Przejście przez wszystkie modele
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii danego modelu
    df.index += 1  # Numerowanie epok od 1

    fig = plt.figure(figsize=(16, 12))  # Utworzenie obszaru rysunku o rozmiarze 16 × 12 cali

    ax = fig.add_subplot(211)  # Dodanie górnego wykresu dokładności
    ax.plot(df['accuracy'], "bp--")  # Dokładność treningowa: niebieska przerywana linia
    ax.plot(df['val_accuracy'], "rp--")  # Dokładność walidacyjna: czerwona przerywana linia
    ax.set_title(f'Model {model} Accuracy', fontsize=20)  # Ustawienie tytułu wykresu
    ax.set_ylabel('Accuracy', fontsize=15)  # Opis osi pionowej
    ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
    ax.set_xlim([1, max_index])  # Ustawienie wspólnego zakresu epok
    ax.set_ylim([min_accuracy, 1])  # Ustawienie zakresu dokładności

    for milestone in (0.7, 0.8, 0.9, 0.95):  # Analiza wybranych progów dokładności
        ax.axhline(milestone, color="k", linestyle="--")  # Pozioma linia oznaczająca próg

        milestone_epochs = df.index[df['val_accuracy'] >= milestone]  # Epoki, w których osiągnięto próg

        if not milestone_epochs.empty:  # Sprawdzenie, czy model osiągnął dany próg
            first_epoch = milestone_epochs.min()  # Pierwsza epoka, w której osiągnięto próg

            if first_epoch > 1:  # Pominięcie pionowej linii, jeśli próg osiągnięto w pierwszej epoce
                ax.axvline(first_epoch, color="g", linestyle="--")  # Zaznaczenie pierwszej odpowiedniej epoki
                ax.text(  # Dodanie opisu numeru epoki
                    first_epoch + 0.6,  # Położenie tekstu na osi poziomej
                    min_accuracy + 0.02,  # Położenie tekstu na osi pionowej
                    f'Epoch: {first_epoch}',  # Treść etykiety
                    rotation=90  # Obrócenie tekstu o 90 stopni
                )

    ax.legend(['Training', 'Validation'], loc='lower right')  # Dodanie legendy dokładności

    ax = fig.add_subplot(212)  # Dodanie dolnego wykresu straty
    ax.plot(df['loss'], "bp--")  # Strata treningowa: niebieska przerywana linia
    ax.plot(df['val_loss'], "rp--")  # Strata walidacyjna: czerwona przerywana linia
    ax.set_title(f'Model {model} Loss', fontsize=20)  # Ustawienie tytułu wykresu
    ax.set_ylabel('Loss', fontsize=15)  # Opis osi pionowej
    ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
    ax.set_xlim([1, max_index])  # Ustawienie wspólnego zakresu epok
    ax.set_ylim([0, max_loss])  # Ustawienie wspólnego zakresu wartości straty
    ax.legend(['Training', 'Validation'], loc='upper right')  # Dodanie legendy straty

    plt.tight_layout()  # Automatyczne dopasowanie odstępów między wykresami
    plt.savefig(  # Zapisanie wykresu do pliku
        f'charts/train_history_{model}.png',  # Nazwa pliku zależna od nazwy modelu
        transparent=True,  # Ustawienie przezroczystego tła
        dpi=600  # Zapis w wysokiej rozdzielczości
    )
    plt.show()  # Wyświetlenie wykresu


# Utworzenie wspólnego wykresu porównującego wszystkie modele
fig = plt.figure(figsize=(16, 12))  # Utworzenie obszaru wspólnego wykresu
ax = fig.add_subplot(211)  # Dodanie górnego wykresu dokładności walidacyjnej

for model, color in zip(models, colors):  # Przejście przez modele i przypisane kolory
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii danego modelu
    df.index += 1  # Numerowanie epok od 1

    ax.plot(  # Narysowanie dokładności walidacyjnej
        df['val_accuracy'],  # Wartości dokładności walidacyjnej
        label=f'Model {model}',  # Nazwa modelu w legendzie
        color=color,  # Kolor przypisany do modelu
        linewidth=3  # Grubość linii
    )

    ax.axhline(  # Zaznaczenie najlepszej dokładności modelu
        df['val_accuracy'].max(),  # Największa dokładność walidacyjna
        color=color,  # Kolor odpowiadający modelowi
        linestyle="dotted",  # Kropkowany styl linii
        linewidth=4  # Grubość linii
    )

ax.set_title('Accuracy', fontsize=20)  # Tytuł wykresu dokładności
ax.set_ylabel('Accuracy', fontsize=15)  # Opis osi pionowej
ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
ax.set_xlim([1, max_index])  # Wspólny zakres epok
ax.set_ylim([min_accuracy, 1])  # Wspólny zakres dokładności

for milestone in (0.7, 0.8, 0.9, 0.95):  # Przejście przez progi dokładności
    ax.axhline(milestone, color="k", linestyle="--")  # Zaznaczenie progów poziomymi liniami

ax.legend(loc='lower right')  # Dodanie legendy modeli


ax = fig.add_subplot(212)  # Dodanie dolnego wykresu straty walidacyjnej

for model, color in zip(models, colors):  # Przejście przez modele i ich kolory
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii danego modelu
    df.index += 1  # Numerowanie epok od 1

    ax.plot(  # Narysowanie straty walidacyjnej
        df['val_loss'],  # Wartości straty walidacyjnej
        label=f'Model {model}',  # Nazwa modelu w legendzie
        color=color,  # Kolor przypisany do modelu
        linewidth=3  # Grubość linii
    )

    ax.axhline(  # Zaznaczenie najmniejszej straty modelu
        df['val_loss'].min(),  # Najmniejsza strata walidacyjna
        color=color,  # Kolor odpowiadający modelowi
        linestyle="dotted",  # Kropkowany styl linii
        linewidth=4  # Grubość linii
    )

ax.set_title('Loss', fontsize=20)  # Tytuł wykresu straty
ax.set_ylabel('Loss', fontsize=15)  # Opis osi pionowej
ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
ax.set_xlim([1, max_index])  # Wspólny zakres epok
ax.set_ylim([0, max_loss])  # Wspólny zakres wartości straty
ax.legend(loc='upper right')  # Dodanie legendy modeli

plt.tight_layout()  # Dopasowanie odstępów między wykresami
plt.savefig(  # Zapisanie wspólnego wykresu
    'charts/train_history_of_each_model.png',  # Nazwa pliku wynikowego
    transparent=True,  # Przezroczyste tło
    dpi=600  # Wysoka rozdzielczość obrazu
)
plt.show()  # Wyświetlenie wspólnego wykresu

Najważniejsze działanie kodu:
- Pierwsza pętla ustala wspólne zakresy osi dla wszystkich modeli.
- Druga pętla tworzy dla każdego modelu wykres dokładności i straty.
- Linie poziome 0.7, 0.8, 0.9 i 0.95 pokazują progi dokładności.
- Zielona linia pionowa wskazuje pierwszą epokę, w której model osiągnął dany próg.
- Ostatnia część porównuje val_accuracy i val_loss wszystkich modeli.
- Linie kropkowane wskazują najlepszą dokładność oraz najmniejszą stratę każdego modelu.
- Wszystkie wykresy są zapisywane w katalogu charts.

Użyłem pd.read_csv(...) bez index_col=0, ponieważ wcześniejszy kod zapisywał pliki za pomocą index=False. Dzięki temu pierwsza kolumna z wynikami nie zostanie omyłkowo użyta jako indeks.

Model ten jest bardzo niedopasowany, ale mimo to częściej ma rację niż rzut monetą. Dlatego, jak sama nazwa wskazuje, zostanie on użyty do porównania.

**Model podstawowy 1**

Ten model będzie prostym modelem sieci konwolucyjnej z jedną warstwą splotową oraz jedną warstwą łączącą.

In [ ]:
model_simple_1 = Sequential()  # Utworzenie sekwencyjnego modelu sieci neuronowej

model_simple_1.add(  # Dodanie warstwy konwolucyjnej
    Conv2D(
        filters=10,  # Zastosowanie 10 filtrów wykrywających różne cechy obrazu
        kernel_size=(3, 3),  # Rozmiar przesuwającego się filtra
        activation='relu',  # Funkcja aktywacji wprowadzająca nieliniowość
        input_shape=(150, 150, 3)  # Obrazy 150 × 150 pikseli z trzema kanałami RGB
    )
)

model_simple_1.add(MaxPooling2D(2, 2))  # Dwukrotne zmniejszenie wysokości i szerokości map cech
model_simple_1.add(Flatten())  # Spłaszczenie map cech do jednowymiarowego wektora
model_simple_1.add(Dense(units=1, activation='sigmoid'))  # Obliczenie prawdopodobieństwa jednej z dwóch klas

model_simple_1.compile(  # Przygotowanie modelu do trenowania
    loss='binary_crossentropy',  # Funkcja straty dla klasyfikacji binarnej
    optimizer='adam',  # Algorytm aktualizujący wagi modelu
    metrics=['accuracy']  # Monitorowanie dokładności podczas uczenia
)

model_simple_1.summary()  # Wyświetlenie architektury i liczby parametrów modelu
models.append("simple_1")  # Dodanie nazwy modelu do listy porównywanych modeli

Model działa następująco:
- Conv2D przesuwa po obrazie 10 filtrów o wymiarach 3 × 3. Filtry uczą się wykrywać między innymi krawędzie, kolory i proste wzory.
- MaxPooling2D zmniejsza mapy cech, zachowując ich najważniejsze wartości.
- Flatten przekształca mapy cech w wektor.
- Dense z funkcją sigmoid zwraca wartość od 0 do 1, interpretowaną jako prawdopodobieństwo klasy pozytywnej.
- Model jest przygotowany do klasyfikacji binarnej za pomocą funkcji straty binary_crossentropy.


Przewidywane wymiary danych:
- wejście: (150, 150, 3),
- po Conv2D: (148, 148, 10),
- po MaxPooling2D: (74, 74, 10),
- po Flatten: 54 760 wartości,
- warstwa Dense: 54 761 parametrów, czyli 54 760 wag i jeden bias.

Przejdźmy do uczenia modelu, jest to tożsame z tym, jak uczony był poprzedni model – podmieniamy jedynie nazwy zmiennych. Dlatego później nie będzie pokazywany ten kod dla kolejnych modeli.

In [ ]:
import os  # Obsługa katalogów i ścieżek

history_simple_1 = model_simple_1.fit(  # Rozpoczęcie trenowania modelu simple_1
    train_generator,  # Generator dostarczający partie danych treningowych
    steps_per_epoch=steps_per_epoch,  # Liczba partii treningowych w jednej epoce
    epochs=epochs,  # Maksymalna liczba epok uczenia
    validation_data=validation_generator,  # Generator danych walidacyjnych
    validation_steps=validation_steps,  # Liczba partii używanych podczas walidacji
    callbacks=[es]  # EarlyStopping zatrzymujący uczenie po braku poprawy
)

history_simple_1_df = pd.DataFrame(history_simple_1.history)  # Zamiana historii uczenia na tabelę

os.makedirs('history', exist_ok=True)  # Utworzenie katalogu history, jeśli nie istnieje

history_simple_1_csv_file = 'history/history_simple_1.csv'  # Ścieżka pliku z historią modelu

history_simple_1_df.to_csv(  # Zapisanie historii uczenia do pliku CSV
    history_simple_1_csv_file,  # Plik docelowy
    index=False  # Pominięcie dodatkowej kolumny z indeksem DataFrame
)

Kod trenuje model model_simple_1, korzystając z generatorów danych. Po każdej epoce zapisuje w obiekcie history_simple_1 takie wyniki jak accuracy, loss, val_accuracy oraz val_loss. Następnie zamienia wyniki na tabelę i zapisuje je w pliku history/history_simple_1.csv

**Model podstawowy 2**

---



Podczas operacji splotu boczne piksele są usuwane, co powoduje utratę informacji. Można tego uniknąć, jeśli obraz wejściowy zostanie otoczony pikselami. W tym modelu jest to wykorzystywane za pomocą wypełnienia - padding.

In [ ]:
model_simple_2 = Sequential()  # Utworzenie sekwencyjnego modelu sieci neuronowej

model_simple_2.add(  # Dodanie warstwy konwolucyjnej
    Conv2D(
        filters=10,  # Zastosowanie 10 filtrów uczących się różnych cech obrazu
        kernel_size=(3, 3),  # Ustawienie rozmiaru każdego filtra na 3 × 3
        padding='same',  # Zachowanie wysokości i szerokości obrazu po konwolucji
        activation='relu',  # Zastosowanie funkcji aktywacji ReLU
        input_shape=(150, 150, 3)  # Obrazy wejściowe 150 × 150 z trzema kanałami RGB
    )
)

model_simple_2.add(MaxPooling2D(2, 2))  # Dwukrotne zmniejszenie wymiarów map cech
model_simple_2.add(Flatten())  # Przekształcenie map cech w jednowymiarowy wektor
model_simple_2.add(Dense(units=1, activation='sigmoid'))  # Zwrócenie prawdopodobieństwa klasy pozytywnej

model_simple_2.compile(  # Przygotowanie modelu do trenowania
    loss='binary_crossentropy',  # Funkcja straty przeznaczona do klasyfikacji binarnej
    optimizer='adam',  # Optymalizator aktualizujący wagi modelu
    metrics=['accuracy']  # Monitorowanie dokładności modelu
)

model_simple_2.summary()  # Wyświetlenie architektury i liczby parametrów modelu
models.append("simple_2")  # Dodanie nazwy modelu do listy porównywanych modeli

To, co istotne – liczba parametrów nieznacznie wzrosła, jednak w tym przypadku nie tracimy informacji z pikseli, które są na krawędziach.

In [ ]:
model_1 = Sequential()  # Utworzenie modelu 1
model_1.add(Conv2D(10, (3, 3), activation='relu',
                   input_shape=(150, 150, 3)))  # Konwolucja bez paddingu
model_1.add(MaxPooling2D((2, 2)))  # Zmniejszenie map cech
model_1.add(Flatten())  # Spłaszczenie map cech
model_1.add(Dense(1, activation='sigmoid'))  # Klasyfikacja binarna

model_1.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


model_2 = Sequential()  # Utworzenie modelu 2
model_2.add(Conv2D(10, (3, 3), padding='same', activation='relu',
                   input_shape=(150, 150, 3)))  # Konwolucja z paddingiem
model_2.add(MaxPooling2D((2, 2)))  # Zmniejszenie map cech
model_2.add(Flatten())  # Spłaszczenie map cech
model_2.add(Dense(1, activation='sigmoid'))  # Klasyfikacja binarna

model_2.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
model_2 = Sequential()  # Utworzenie drugiego modelu konwolucyjnego

# Blok konwolucyjny 1
model_2.add(Conv2D(
    filters=16,  # Liczba filtrów
    kernel_size=(3, 3),  # Rozmiar każdego filtra
    activation='relu',  # Funkcja aktywacji ReLU
    padding='same',  # Zachowanie wymiarów map cech
    input_shape=(150, 150, 3)  # Obrazy wejściowe RGB o wymiarach 150 × 150
))
model_2.add(Conv2D(filters=16, kernel_size=(3, 3),
                   activation='relu', padding='same'))  # Druga konwolucja z 16 filtrami
model_2.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech do 75 × 75

# Blok konwolucyjny 2
model_2.add(Conv2D(filters=32, kernel_size=(3, 3),
                   activation='relu', padding='same'))  # Konwolucja z 32 filtrami
model_2.add(Conv2D(filters=32, kernel_size=(3, 3),
                   activation='relu', padding='same'))  # Druga konwolucja z 32 filtrami
model_2.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech do 37 × 37

# Blok konwolucyjny 3
model_2.add(Conv2D(filters=64, kernel_size=(3, 3),
                   activation='relu', padding='same'))  # Konwolucja z 64 filtrami
model_2.add(Conv2D(filters=64, kernel_size=(3, 3),
                   activation='relu', padding='same'))  # Druga konwolucja z 64 filtrami
model_2.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech do 18 × 18

# Blok konwolucyjny 4
model_2.add(Conv2D(filters=128, kernel_size=(3, 3),
                   activation='relu', padding='same'))  # Konwolucja ze 128 filtrami
model_2.add(Conv2D(filters=128, kernel_size=(3, 3),
                   activation='relu', padding='same'))  # Druga konwolucja ze 128 filtrami
model_2.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech do 9 × 9

# Część klasyfikacyjna
model_2.add(Flatten())  # Spłaszczenie map cech do wektora
model_2.add(Dropout(0.5))  # Losowe wyłączenie 50% wartości wektora podczas uczenia
model_2.add(Dense(256, activation='relu'))  # Warstwa ukryta z 256 neuronami
model_2.add(Dropout(0.5))  # Ponowne wyłączenie 50% neuronów podczas uczenia
model_2.add(Dense(128, activation='relu'))  # Warstwa ukryta ze 128 neuronami
model_2.add(Dense(units=1, activation='sigmoid'))  # Wyjście klasyfikacji binarnej

model_2.compile(  # Przygotowanie modelu do trenowania
    loss='binary_crossentropy',  # Funkcja straty dla klasyfikacji binarnej
    optimizer=RMSprop(learning_rate=1e-4),  # Optymalizator z tempem uczenia 0,0001
    metrics=['accuracy']  # Monitorowanie dokładności
)

model_2.summary()  # Wyświetlenie architektury i liczby parametrów

if 'model_2' not in models:  # Zabezpieczenie przed powtórzeniem nazwy
    models.append('model_2')  # Dodanie nazwy do listy modeli

In [ ]:
import os  # Obsługa katalogów i ścieżek

os.makedirs('history', exist_ok=True)  # Utworzenie katalogu history, jeśli nie istnieje

history_simple_2 = model_simple_2.fit(  # Trenowanie modelu simple_2
    train_generator,  # Generator danych treningowych
    steps_per_epoch=steps_per_epoch,  # Liczba partii treningowych w jednej epoce
    epochs=epochs,  # Maksymalna liczba epok
    validation_data=validation_generator,  # Generator danych walidacyjnych
    validation_steps=validation_steps,  # Liczba partii walidacyjnych
    callbacks=[es]  # Zastosowanie mechanizmu EarlyStopping
)

history_simple_2_df = pd.DataFrame(history_simple_2.history)  # Utworzenie tabeli z historią uczenia

history_simple_2_csv_file = 'history/history_simple_2.csv'  # Ścieżka pliku wynikowego

history_simple_2_df.to_csv(  # Zapisanie wyników uczenia
    history_simple_2_csv_file,  # Docelowy plik CSV
    index=False  # Zapis bez dodatkowej kolumny indeksu
)

In [ ]:
import os  # Obsługa katalogów
import numpy as np  # Operacje numeryczne
import pandas as pd  # Odczytywanie plików CSV
import matplotlib.pyplot as plt  # Tworzenie wykresów

os.makedirs('charts', exist_ok=True)  # Utworzenie katalogu na wykresy, jeśli nie istnieje

max_index = 0  # Największa liczba epok spośród wszystkich modeli
min_accuracy = 1  # Najmniejsza dokładność spośród wszystkich modeli
max_loss = 0  # Największa wartość straty spośród wszystkich modeli
colors = plt.cm.rainbow(np.linspace(0, 1, len(models)))  # Wygenerowanie osobnego koloru dla każdego modelu

# Ustalenie wspólnych zakresów osi dla wszystkich wykresów
for model in models:  # Przejście przez nazwy wszystkich modeli
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii uczenia danego modelu
    df.index += 1  # Numerowanie epok od 1 zamiast od 0

    if max_index < df.index.max():  # Sprawdzenie, czy model trenował dłużej niż poprzednie
        max_index = df.index.max()  # Zapisanie największej liczby epok

    current_min_accuracy = df[['accuracy', 'val_accuracy']].min().min()  # Najmniejsza dokładność modelu
    if min_accuracy > current_min_accuracy:  # Sprawdzenie, czy znaleziono mniejszą dokładność
        min_accuracy = current_min_accuracy  # Aktualizacja dolnej granicy wykresów dokładności

    current_max_loss = df[['loss', 'val_loss']].max().max()  # Największa strata modelu
    if max_loss < current_max_loss:  # Sprawdzenie, czy znaleziono większą stratę
        max_loss = current_max_loss  # Aktualizacja górnej granicy wykresów straty


# Utworzenie osobnych wykresów dla każdego modelu
for model in models:  # Przejście przez wszystkie modele
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii danego modelu
    df.index += 1  # Numerowanie epok od 1

    fig = plt.figure(figsize=(16, 12))  # Utworzenie obszaru rysunku o rozmiarze 16 × 12 cali

    ax = fig.add_subplot(211)  # Dodanie górnego wykresu dokładności
    ax.plot(df['accuracy'], "bp--")  # Dokładność treningowa: niebieska przerywana linia
    ax.plot(df['val_accuracy'], "rp--")  # Dokładność walidacyjna: czerwona przerywana linia
    ax.set_title(f'Model {model} Accuracy', fontsize=20)  # Ustawienie tytułu wykresu
    ax.set_ylabel('Accuracy', fontsize=15)  # Opis osi pionowej
    ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
    ax.set_xlim([1, max_index])  # Ustawienie wspólnego zakresu epok
    ax.set_ylim([min_accuracy, 1])  # Ustawienie zakresu dokładności

    for milestone in (0.7, 0.8, 0.9, 0.95):  # Analiza wybranych progów dokładności
        ax.axhline(milestone, color="k", linestyle="--")  # Pozioma linia oznaczająca próg

        milestone_epochs = df.index[df['val_accuracy'] >= milestone]  # Epoki, w których osiągnięto próg

        if not milestone_epochs.empty:  # Sprawdzenie, czy model osiągnął dany próg
            first_epoch = milestone_epochs.min()  # Pierwsza epoka, w której osiągnięto próg

            if first_epoch > 1:  # Pominięcie pionowej linii, jeśli próg osiągnięto w pierwszej epoce
                ax.axvline(first_epoch, color="g", linestyle="--")  # Zaznaczenie pierwszej odpowiedniej epoki
                ax.text(  # Dodanie opisu numeru epoki
                    first_epoch + 0.6,  # Położenie tekstu na osi poziomej
                    min_accuracy + 0.02,  # Położenie tekstu na osi pionowej
                    f'Epoch: {first_epoch}',  # Treść etykiety
                    rotation=90  # Obrócenie tekstu o 90 stopni
                )

    ax.legend(['Training', 'Validation'], loc='lower right')  # Dodanie legendy dokładności

    ax = fig.add_subplot(212)  # Dodanie dolnego wykresu straty
    ax.plot(df['loss'], "bp--")  # Strata treningowa: niebieska przerywana linia
    ax.plot(df['val_loss'], "rp--")  # Strata walidacyjna: czerwona przerywana linia
    ax.set_title(f'Model {model} Loss', fontsize=20)  # Ustawienie tytułu wykresu
    ax.set_ylabel('Loss', fontsize=15)  # Opis osi pionowej
    ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
    ax.set_xlim([1, max_index])  # Ustawienie wspólnego zakresu epok
    ax.set_ylim([0, max_loss])  # Ustawienie wspólnego zakresu wartości straty
    ax.legend(['Training', 'Validation'], loc='upper right')  # Dodanie legendy straty

    plt.tight_layout()  # Automatyczne dopasowanie odstępów między wykresami
    plt.savefig(  # Zapisanie wykresu do pliku
        f'charts/train_history_{model}.png',  # Nazwa pliku zależna od nazwy modelu
        transparent=True,  # Ustawienie przezroczystego tła
        dpi=600  # Zapis w wysokiej rozdzielczości
    )
    plt.show()  # Wyświetlenie wykresu


# Utworzenie wspólnego wykresu porównującego wszystkie modele
fig = plt.figure(figsize=(16, 12))  # Utworzenie obszaru wspólnego wykresu
ax = fig.add_subplot(211)  # Dodanie górnego wykresu dokładności walidacyjnej

for model, color in zip(models, colors):  # Przejście przez modele i przypisane kolory
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii danego modelu
    df.index += 1  # Numerowanie epok od 1

    ax.plot(  # Narysowanie dokładności walidacyjnej
        df['val_accuracy'],  # Wartości dokładności walidacyjnej
        label=f'Model {model}',  # Nazwa modelu w legendzie
        color=color,  # Kolor przypisany do modelu
        linewidth=3  # Grubość linii
    )

    ax.axhline(  # Zaznaczenie najlepszej dokładności modelu
        df['val_accuracy'].max(),  # Największa dokładność walidacyjna
        color=color,  # Kolor odpowiadający modelowi
        linestyle="dotted",  # Kropkowany styl linii
        linewidth=4  # Grubość linii
    )

ax.set_title('Accuracy', fontsize=20)  # Tytuł wykresu dokładności
ax.set_ylabel('Accuracy', fontsize=15)  # Opis osi pionowej
ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
ax.set_xlim([1, max_index])  # Wspólny zakres epok
ax.set_ylim([min_accuracy, 1])  # Wspólny zakres dokładności

for milestone in (0.7, 0.8, 0.9, 0.95):  # Przejście przez progi dokładności
    ax.axhline(milestone, color="k", linestyle="--")  # Zaznaczenie progów poziomymi liniami

ax.legend(loc='lower right')  # Dodanie legendy modeli


ax = fig.add_subplot(212)  # Dodanie dolnego wykresu straty walidacyjnej

for model, color in zip(models, colors):  # Przejście przez modele i ich kolory
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii danego modelu
    df.index += 1  # Numerowanie epok od 1

    ax.plot(  # Narysowanie straty walidacyjnej
        df['val_loss'],  # Wartości straty walidacyjnej
        label=f'Model {model}',  # Nazwa modelu w legendzie
        color=color,  # Kolor przypisany do modelu
        linewidth=3  # Grubość linii
    )

    ax.axhline(  # Zaznaczenie najmniejszej straty modelu
        df['val_loss'].min(),  # Najmniejsza strata walidacyjna
        color=color,  # Kolor odpowiadający modelowi
        linestyle="dotted",  # Kropkowany styl linii
        linewidth=4  # Grubość linii
    )

ax.set_title('Loss', fontsize=20)  # Tytuł wykresu straty
ax.set_ylabel('Loss', fontsize=15)  # Opis osi pionowej
ax.set_xlabel('Epoch', fontsize=15)  # Opis osi poziomej
ax.set_xlim([1, max_index])  # Wspólny zakres epok
ax.set_ylim([0, max_loss])  # Wspólny zakres wartości straty
ax.legend(loc='upper right')  # Dodanie legendy modeli

plt.tight_layout()  # Dopasowanie odstępów między wykresami
plt.savefig(  # Zapisanie wspólnego wykresu
    'charts/train_history_of_each_model.png',  # Nazwa pliku wynikowego
    transparent=True,  # Przezroczyste tło
    dpi=600  # Wysoka rozdzielczość obrazu
)
plt.show()  # Wyświetlenie wspólnego wykresu

**Model 3**

---



Mimo wykorzystania techniki porzucenia, model w końcowej fazie się nie uczy uogólnionych wzorców, tylko się przeucza i uczy się zdjęć ze zbioru treningowego. Teraz rozszerzymy zbiór treningowy poprzez augmentację danych.

In [ ]:
model_3 = Sequential()  # Utworzenie sekwencyjnego modelu sieci konwolucyjnej

# Blok konwolucyjny 1
model_3.add(Conv2D(filters=16, kernel_size=(3, 3), activation='relu', padding='same',
                   input_shape=(150, 150, 3)))  # Wykrywanie podstawowych cech w obrazach RGB 150 × 150
model_3.add(Conv2D(filters=16, kernel_size=(3, 3), activation='relu',
                   padding='same'))  # Dalsze przetwarzanie cech za pomocą 16 filtrów
model_3.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech z 150 × 150 do 75 × 75

# Blok konwolucyjny 2
model_3.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu',
                   padding='same'))  # Wykrywanie bardziej złożonych cech za pomocą 32 filtrów
model_3.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu',
                   padding='same'))  # Dalsze przetwarzanie cech w drugim bloku
model_3.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech z 75 × 75 do 37 × 37

# Blok konwolucyjny 3
model_3.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu',
                   padding='same'))  # Wykrywanie bardziej szczegółowych cech za pomocą 64 filtrów
model_3.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu',
                   padding='same'))  # Dalsze przetwarzanie cech w trzecim bloku
model_3.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech z 37 × 37 do 18 × 18

# Blok konwolucyjny 4
model_3.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu',
                   padding='same'))  # Wykrywanie złożonych cech za pomocą 128 filtrów
model_3.add(Conv2D(filters=128, kernel_size=(3, 3), activation='relu',
                   padding='same'))  # Dalsze przetwarzanie cech w czwartym bloku
model_3.add(MaxPooling2D(pool_size=(2, 2)))  # Zmniejszenie map cech z 18 × 18 do 9 × 9

model_3.add(Flatten())  # Spłaszczenie map cech 9 × 9 × 128 do jednego wektora
model_3.add(Dropout(0.5))  # Losowe wyłączenie 50% neuronów w celu ograniczenia przeuczenia
model_3.add(Dense(256, activation='relu'))  # Warstwa ukryta zawierająca 256 neuronów
model_3.add(Dropout(0.5))  # Ponowne wyłączenie 50% neuronów podczas trenowania
model_3.add(Dense(128, activation='relu'))  # Warstwa ukryta zawierająca 128 neuronów
model_3.add(Dense(units=1, activation='sigmoid'))  # Wyjście dla klasyfikacji binarnej

model_3.compile(  # Przygotowanie modelu do trenowania
    loss='binary_crossentropy',  # Funkcja straty dla klasyfikacji dwóch klas
    optimizer=RMSprop(learning_rate=1e-4),  # Optymalizator z tempem uczenia 0,0001
    metrics=['accuracy']  # Monitorowanie dokładności modelu
)

model_3.summary()  # Wyświetlenie architektury i liczby parametrów modelu
models.append("model_3")  # Dodanie nazwy modelu do listy porównywanych modeli


In [ ]:
import os  # Obsługa katalogów i ścieżek

os.makedirs('history', exist_ok=True)  # Utworzenie katalogu history, jeśli nie istnieje

history_model_3 = model_3.fit(  # Rozpoczęcie trenowania modelu model_3
    train_generator_augmentation,  # Generator treningowy z augmentacją obrazów
    steps_per_epoch=steps_per_epoch,  # Liczba partii treningowych w jednej epoce
    epochs=epochs,  # Maksymalna liczba epok uczenia
    validation_data=validation_generator,  # Generator danych walidacyjnych bez augmentacji
    validation_steps=validation_steps,  # Liczba partii walidacyjnych w jednej epoce
    callbacks=[es]  # Zastosowanie mechanizmu EarlyStopping
)

history_model_3_df = pd.DataFrame(history_model_3.history)  # Zamiana historii uczenia na tabelę

history_model_3_csv_file = 'history/history_model_3.csv'  # Określenie ścieżki pliku wynikowego

history_model_3_df.to_csv(  # Zapisanie historii trenowania do pliku CSV
    history_model_3_csv_file,  # Ścieżka docelowego pliku
    index=False  # Zapis bez dodatkowej kolumny indeksu
)

Wyniki charakteryzrują się większą zmiennością, ponieważ za każdym razem zbiór treningowy był inny. Co więcej, wyniki były często lepsze dla zbioru walidacyjnego dzięki zastosowaniu różnych zestawów treningowych. Ponownie wydłużyliśmy czas uczenia się modeli, dodając kolejną technikę regularyzacji. Niemniej wyniki stały się znowu lepsze. Dodając kolejne ‘klocki’ zdecydowanie polepszyliśmy jakość modelu, porównaj sobie wyniki tej sieci z pierwszą wytrenowaną siecią.

Model 4

---



Ostatnim klockiem do dodania został transfer learning. Jako podstawę wykorzystamy model VGG16, który został wyuczony na podstawie zbioru danych ImageNet (liczącej ponad 14 milionów zdjęć z 20 tysiącami klas). Trenowane są trzy ostatnie warstwy splotowe oraz warstwy w pełni połączone. Stosujemy również pozostałe wykorzystane wcześniej techniki regularyzacji, jak Dropout czy data augmentation.

In [ ]:
from tensorflow.keras.applications import VGG16  # Import gotowego modelu VGG16

vgg16 = VGG16(  # Wczytanie modelu VGG16
    weights='imagenet',  # Użycie wag wytrenowanych na zbiorze ImageNet
    include_top=False,  # Pominięcie oryginalnych warstw klasyfikacyjnych VGG16
    input_shape=(150, 150, 3)  # Ustawienie rozmiaru obrazów wejściowych RGB
)

vgg16.trainable = True  # Zezwolenie na indywidualne ustawianie warstw jako trenowalne

set_trainable = False  # Początkowo warstwy modelu pozostają zamrożone

for layer in vgg16.layers:  # Przejście przez wszystkie warstwy modelu VGG16
    if layer.name == 'block5_conv1':  # Sprawdzenie, czy osiągnięto pierwszy splot piątego bloku
        set_trainable = True  # Od tego miejsca warstwy będą trenowane

    if set_trainable:  # Jeśli osiągnięto blok piąty
        layer.trainable = True  # Odblokowanie warstwy i aktualizowanie jej wag
    else:  # Jeśli warstwa znajduje się przed piątym blokiem
        layer.trainable = False  # Zamrożenie wcześniej wyuczonych wag warstwy

for layer in vgg16.layers:  # Ponowne przejście przez wszystkie warstwy
    print(  # Wyświetlenie informacji o warstwie
        f'layer_name: {layer.name:13} trainable: {layer.trainable}'  # Nazwa i status warstwy
    )

model_4 = Sequential()  # Utworzenie sekwencyjnego modelu klasyfikacyjnego
model_4.add(vgg16)  # Dodanie VGG16 jako ekstraktora cech obrazu
model_4.add(Flatten())  # Spłaszczenie map cech do jednowymiarowego wektora
model_4.add(Dropout(0.5))  # Wyłączenie 50% neuronów podczas uczenia
model_4.add(Dense(256, activation='relu'))  # Dodanie warstwy ukrytej z 256 neuronami
model_4.add(Dropout(0.5))  # Ograniczenie przeuczenia modelu
model_4.add(Dense(128, activation='relu'))  # Dodanie warstwy ukrytej ze 128 neuronami
model_4.add(Dense(units=1, activation='sigmoid'))  # Wyjście dla klasyfikacji binarnej

model_4.compile(  # Przygotowanie modelu do trenowania
    loss='binary_crossentropy',  # Funkcja straty dla klasyfikacji binarnej
    optimizer=RMSprop(learning_rate=1e-4),  # Optymalizator z małym tempem uczenia
    metrics=['accuracy']  # Monitorowanie dokładności podczas trenowania
)

model_4.summary()  # Wyświetlenie architektury i liczby parametrów modelu
models.append("model_4")  # Dodanie nazwy modelu do listy porównywanych modeli

Model ma wszystkie zalety poprzedniego modelu oraz to, co istotne, nie uczył się od początku. Dokładność 90% została osiągnięta po drugiej epoce, podczas gdy poprzedni model osiągnął ten wynik po siedemdziesiątej szóstej epoce.

In [ ]:
print(models)  # Modele oczekiwane przez kod wykresów
print(os.listdir('history'))  # Pliki znajdujące się w katalogu history

In [ ]:
history_model_4 = model_4.fit(  # Trenowanie modelu VGG16
    train_generator_augmentation,  # Dane treningowe z augmentacją
    steps_per_epoch=steps_per_epoch,  # Liczba partii w jednej epoce
    epochs=epochs,  # Maksymalna liczba epok
    validation_data=validation_generator,  # Dane walidacyjne
    validation_steps=validation_steps,  # Liczba partii walidacyjnych
    callbacks=[es]  # Mechanizm EarlyStopping
)

history_model_4_df = pd.DataFrame(history_model_4.history)  # Historia wyników uczenia

history_model_4_df.to_csv(  # Zapis historii do pliku
    'history/history_model_4.csv',  # Oczekiwana nazwa pliku
    index=False  # Zapis bez dodatkowego indeksu
)

In [ ]:
print(os.listdir('history'))  # Wyświetlenie zapisanych plików

In [ ]:
import os  # Obsługa katalogów
import numpy as np  # Operacje numeryczne
import pandas as pd  # Odczytywanie plików CSV
import matplotlib.pyplot as plt  # Tworzenie wykresów

os.makedirs('charts', exist_ok=True)  # Utworzenie katalogu na wykresy

max_index = 0  # Największa liczba epok spośród wszystkich modeli
min_accuracy = 1  # Najmniejsza dokładność spośród wszystkich modeli
max_loss = 0  # Największa strata spośród wszystkich modeli
colors = plt.cm.rainbow(np.linspace(0, 1, len(models)))  # Osobny kolor dla każdego modelu


# Ustalenie wspólnych zakresów osi
for model in models:  # Przejście przez wszystkie modele
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii modelu
    df.index += 1  # Numerowanie epok od 1

    if max_index < df.index.max():  # Sprawdzenie liczby epok danego modelu
        max_index = df.index.max()  # Zapisanie największej liczby epok

    current_min_accuracy = df[['accuracy', 'val_accuracy']].min().min()  # Najmniejsza dokładność modelu

    if min_accuracy > current_min_accuracy:  # Sprawdzenie dolnej granicy dokładności
        min_accuracy = current_min_accuracy  # Aktualizacja najmniejszej dokładności

    current_max_loss = df[['loss', 'val_loss']].max().max()  # Największa strata modelu

    if max_loss < current_max_loss:  # Sprawdzenie górnej granicy straty
        max_loss = current_max_loss  # Aktualizacja największej straty


# Utworzenie osobnych wykresów dla każdego modelu
for model in models:  # Przejście przez wszystkie modele
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii modelu
    df.index += 1  # Numerowanie epok od 1

    fig = plt.figure(figsize=(16, 12))  # Utworzenie obszaru wykresu

    ax = fig.add_subplot(211)  # Dodanie górnego wykresu dokładności
    ax.plot(df.index, df['accuracy'], "bp--")  # Dokładność treningowa
    ax.plot(df.index, df['val_accuracy'], "rp--")  # Dokładność walidacyjna
    ax.set_title(f'Model {model} Accuracy', fontsize=20)  # Tytuł wykresu
    ax.set_ylabel('Accuracy', fontsize=15)  # Opis osi Y
    ax.set_xlabel('Epoch', fontsize=15)  # Opis osi X
    ax.set_xlim([1, max_index])  # Automatyczny zakres osi X
    ax.set_ylim([min_accuracy, 1])  # Wspólny zakres dokładności

    for milestone in (0.7, 0.8, 0.9, 0.95):  # Wybrane progi dokładności
        ax.axhline(milestone, color='k', linestyle='--')  # Pozioma linia progu

        milestone_epochs = df.index[df['val_accuracy'] >= milestone]  # Epoki osiągające próg

        if not milestone_epochs.empty:  # Sprawdzenie, czy próg został osiągnięty
            first_epoch = milestone_epochs.min()  # Pierwsza epoka osiągająca próg

            if first_epoch > 1:  # Pominięcie pierwszej epoki
                ax.axvline(first_epoch, color='g', linestyle='--')  # Zaznaczenie epoki

                ax.text(
                    first_epoch + 0.6,  # Położenie tekstu na osi X
                    min_accuracy + 0.02,  # Położenie tekstu na osi Y
                    f'Epoch: {first_epoch}',  # Treść etykiety
                    rotation=90  # Obrócenie tekstu
                )

    ax.legend(['Training', 'Validation'], loc='lower right')  # Legenda dokładności

    ax = fig.add_subplot(212)  # Dodanie dolnego wykresu straty
    ax.plot(df.index, df['loss'], "bp--")  # Strata treningowa
    ax.plot(df.index, df['val_loss'], "rp--")  # Strata walidacyjna
    ax.set_title(f'Model {model} Loss', fontsize=20)  # Tytuł wykresu
    ax.set_ylabel('Loss', fontsize=15)  # Opis osi Y
    ax.set_xlabel('Epoch', fontsize=15)  # Opis osi X
    ax.set_xlim([1, max_index])  # Automatyczny zakres osi X
    ax.set_ylim([0, max_loss])  # Wspólny zakres straty
    ax.legend(['Training', 'Validation'], loc='upper right')  # Legenda straty

    plt.tight_layout()  # Dopasowanie odstępów

    plt.savefig(
        f'charts/train_history_{model}.png',  # Nazwa pliku wykresu
        transparent=True,  # Przezroczyste tło
        dpi=600  # Wysoka rozdzielczość
    )

    plt.show()  # Wyświetlenie wykresu


# Utworzenie wspólnego wykresu porównującego wszystkie modele
fig = plt.figure(figsize=(16, 12))  # Utworzenie obszaru wykresu

ax = fig.add_subplot(211)  # Dodanie wykresu dokładności walidacyjnej

for model, color in zip(models, colors):  # Przejście przez modele i kolory
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii modelu
    df.index += 1  # Numerowanie epok od 1

    ax.plot(
        df.index,  # Numery epok
        df['val_accuracy'],  # Dokładność walidacyjna
        label=f'Model {model}',  # Nazwa modelu
        color=color,  # Kolor linii
        linewidth=3  # Grubość linii
    )

    ax.axhline(
        df['val_accuracy'].max(),  # Największa dokładność walidacyjna
        color=color,  # Kolor modelu
        linestyle='dotted',  # Kropkowana linia
        linewidth=4  # Grubość linii
    )

ax.set_title('Accuracy', fontsize=20)  # Tytuł wykresu
ax.set_ylabel('Accuracy', fontsize=15)  # Opis osi Y
ax.set_xlabel('Epoch', fontsize=15)  # Opis osi X
ax.set_xlim([1, max_index])  # Automatyczny zakres osi X
ax.set_ylim([min_accuracy, 1])  # Wspólny zakres dokładności

for milestone in (0.7, 0.8, 0.9, 0.95):  # Wybrane progi dokładności
    ax.axhline(milestone, color='k', linestyle='--')  # Zaznaczenie progu

ax.legend(loc='lower right')  # Legenda modeli


ax = fig.add_subplot(212)  # Dodanie wykresu straty walidacyjnej

for model, color in zip(models, colors):  # Przejście przez modele i kolory
    df = pd.read_csv(f'history/history_{model}.csv')  # Odczytanie historii modelu
    df.index += 1  # Numerowanie epok od 1

    ax.plot(
        df.index,  # Numery epok
        df['val_loss'],  # Strata walidacyjna
        label=f'Model {model}',  # Nazwa modelu
        color=color,  # Kolor linii
        linewidth=3  # Grubość linii
    )

    ax.axhline(
        df['val_loss'].min(),  # Najmniejsza strata walidacyjna
        color=color,  # Kolor modelu
        linestyle='dotted',  # Kropkowana linia
        linewidth=4  # Grubość linii
    )

ax.set_title('Loss', fontsize=20)  # Tytuł wykresu
ax.set_ylabel('Loss', fontsize=15)  # Opis osi Y
ax.set_xlabel('Epoch', fontsize=15)  # Opis osi X
ax.set_xlim([1, max_index])  # Automatyczny zakres osi X
ax.set_ylim([0, max_loss])  # Wspólny zakres straty
ax.legend(loc='upper right')  # Legenda modeli

plt.tight_layout()  # Dopasowanie odstępów

plt.savefig(
    'charts/train_history_of_each_model.png',  # Nazwa pliku wynikowego
    transparent=True,  # Przezroczyste tło
    dpi=600  # Wysoka rozdzielczość
)

plt.show()  # Wyświetlenie wspólnego wykresu

In [ ]:
import numpy as np  # Operacje numeryczne
import pandas as pd  # Tworzenie tabeli wyników

from sklearn.metrics import precision_score  # Obliczanie precyzji
from sklearn.metrics import recall_score  # Obliczanie czułości
from sklearn.metrics import f1_score  # Obliczanie wyniku F1
from sklearn.metrics import accuracy_score  # Obliczanie dokładności
from sklearn.metrics import roc_auc_score  # Obliczanie pola pod krzywą ROC


model_objects = {  # Powiązanie nazw modeli z obiektami modeli
    'baseline': model_baseline,  # Model bazowy
    'simple_1': model_simple_1,  # Pierwszy prosty model CNN
    'simple_2': model_simple_2,  # Drugi prosty model CNN
    'model_3': model_3,  # Rozbudowany model CNN
    'model_4': model_4  # Model wykorzystujący VGG16
}

results = []  # Lista przechowująca wyniki modeli
y_true = validation_generator.classes  # Rzeczywiste klasy danych walidacyjnych


for model_name, model in model_objects.items():  # Przejście przez wszystkie modele
    validation_generator.reset()  # Ustawienie generatora na początku zbioru

    y_probability = model.predict(  # Obliczenie prawdopodobieństw klas
        validation_generator,  # Generator danych walidacyjnych
        verbose=1  # Wyświetlanie postępu predykcji
    ).ravel()  # Zamiana wyników na jednowymiarową tablicę

    y_probability = y_probability[:len(y_true)]  # Dopasowanie liczby predykcji do liczby etykiet
    y_pred = (y_probability >= 0.5).astype(int)  # Zamiana prawdopodobieństw na klasy 0 lub 1

    precision = precision_score(  # Obliczenie precyzji
        y_true,
        y_pred,
        zero_division=0  # Zwrócenie zera, gdy nie można obliczyć precyzji
    )

    recall = recall_score(  # Obliczenie czułości
        y_true,
        y_pred,
        zero_division=0  # Zwrócenie zera, gdy nie można obliczyć czułości
    )

    f1 = f1_score(  # Obliczenie wyniku F1
        y_true,
        y_pred,
        zero_division=0  # Zwrócenie zera, gdy nie można obliczyć F1
    )

    accuracy = accuracy_score(y_true, y_pred)  # Obliczenie dokładności
    roc_auc = roc_auc_score(y_true, y_probability)  # Obliczenie ROC AUC z prawdopodobieństw

    results.append({  # Dodanie wyników modelu do listy
        'Model': model_name,  # Nazwa modelu
        'Precision': precision,  # Precyzja
        'Recall': recall,  # Czułość
        'F1-score': f1,  # Wynik F1
        'Accuracy': accuracy,  # Dokładność
        'ROC AUC': roc_auc  # Pole pod krzywą ROC
    })


results_df = pd.DataFrame(results)  # Zamiana wyników na tabelę
results_df = results_df.set_index('Model')  # Ustawienie nazw modeli jako indeksu
results_df = results_df.round(4)  # Zaokrąglenie wyników do czterech miejsc
results_df  # Wyświetlenie tabeli

Zbiór testowy, jak i poprzednio zastosowane zbiory, są zbalansowane, zatem wyniki są skorelowane z przedstawionymi wcześniej wykresami uczenia dla zbioru walidacyjnego. Wytrenowane sieci poprawnie nauczyły się obiektów na zdjęciach, osiągając przy tym wysokie wyniki klasyfikacji obrazów dla zbioru testowego. W każdym przypadku duża ilość danych służy jako zbiór do uczenia modelu. Za pomocą augmentacji danych zbiór ten jest rozszerzany. Drugą zastosowaną metodą była metoda Dropout, która sprawia, że każdy neuron jest ważny. Sieć neuronowa sama uczy się wzorców do odpowiedniej klasyfikacji elementów i dowiaduje się, co jest ważne w obrazie. Za rozpoznawanie wzorców odpowiedzialne są filtry (kernele), które wyciągają informacje. Parametrami sieci są wartości w filtrach. Jeśli przed warstwą splotu znajduje się obiekt więcej niż dwuwymiarowy, to filtr ma kształt sześcianu o głębokości 3 dla obrazów RGB lub równej liczbie filtrów w poprzedniej warstwie. Funkcja aktywacji jest używana na wyjściu warstwy splotowej, zwykle jest to ReLU. Jeśli filtr rozpoznaje wzorzec, wartość jest przepuszczana. Z drugiej strony, jeśli wartość nie pasuje do filtru (wartość mniejsza niż 0), zwracana jest wartość 0. Za agregację informacji odpowiedzialna jest warstwa łącząca (pooling layer). Sieci wielowarstwowe naprzemiennie wykorzystują warstwy splotowe (convolution layer) i łączące (pooling layer). Na początku sieci wykrywają proste cechy, aby w późniejszych warstwach wykryć cechy złożone. Interpretacja tego w kontekście klasyfikacji kluczy mogłaby wyglądać następująco:

- Pierwsza warstwa - rozpoznawanie kształtu.
- Warstwa druga - wykrywanie konturów, typu profilu, zęby klucza.
- Trzecia warstwa - klasyfikacja typu klucza.
- Ostatnia warstwa jest rozwijana do postaci wektora, który propaguje się przez głębokie warstwy aż do osiągnięcia predykcji.

Zastosowano technikę wykorzystania wyuczonej sieci (Transfer Learning). Filtry odpowiedzialne za rozpoznawanie podstawowych cech zostały zamrożone. Ostatnie warstwy i warstwy ukryte były uczone, aby wykorzystać sieć do konkretnego problemu.

Regularyzacja jest sposobem na uniknięcie przetrenowania. Dodatkowo ma ona pozytywny wpływ na dokładność sieci.

Rozwinięciem neuronowej sieci konwolucyjnej jest metoda YOLO (You Only Look Once), która jednocześnie klasyfikuje i lokalizuje obiekty na zdjęciach.

